In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [4]:
original_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print(original_df.shape)
print(test_df.shape)

(9864, 19)
(2466, 18)


In [5]:
print(original_df.head())
print(original_df.info())

   Session_ID  Administrative  Administrative_Duration  Informational  \
0      112163               0                     0.00              0   
1      107490               0                     0.00              0   
2      106273               4                    48.80              0   
3      110651               0                     0.00              0   
4      101259               7                   110.25              0   

   Informational_Duration  ProductRelated  ProductRelated_Duration  \
0                     0.0               3                44.500000   
1                     0.0              12               460.200000   
2                     0.0              11               344.800000   
3                     0.0              23               517.035714   
4                     0.0              20               266.583333   

   BounceRates  ExitRates  PageValues  SpecialDay Month  OperatingSystems  \
0     0.066667   0.133333    0.000000         0.0   Dec        

In [6]:
print(original_df.isnull().sum())

Session_ID                   0
Administrative               0
Administrative_Duration    492
Informational                0
Informational_Duration       0
ProductRelated               0
ProductRelated_Duration      0
BounceRates                  0
ExitRates                  691
PageValues                   0
SpecialDay                   0
Month                        0
OperatingSystems             0
Browser                      0
Region                     591
TrafficType                789
VisitorType                394
Weekend                      0
Revenue                      0
dtype: int64


In [7]:
X = original_df.drop("Revenue", axis=1)
y = original_df["Revenue"]

print(X.shape)
print(y.shape)

(9864, 18)
(9864,)


In [8]:
X = X.drop("Session_ID", axis=1)

test_ids = test_df["Session_ID"]
X_test = test_df.drop("Session_ID", axis=1)

print(X.shape)
print(X_test.shape)

(9864, 17)
(2466, 17)


In [9]:
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(exclude=np.number).columns

for col in num_cols:
    X[col] = X[col].fillna(X[col].median())
    X_test[col] = X_test[col].fillna(X[col].median())

for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])
    X_test[col] = X_test[col].fillna(X[col].mode()[0])

print(X.isnull().sum().sum())
print(X_test.isnull().sum().sum())

0
0


In [10]:
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

X_test = X_test.reindex(columns=X.columns, fill_value=0)

print(X.shape)
print(X_test.shape)

(9864, 26)
(2466, 26)


In [11]:
scaler = StandardScaler()

X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

print(X.shape)
print(X_test.shape)

(9864, 26)
(2466, 26)


In [12]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_val.shape)

(7891, 26)
(1973, 26)


In [13]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [14]:
y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)

print("Validation Accuracy:", accuracy)
print("Validation Accuracy (%):", accuracy * 100)

Validation Accuracy: 0.8677141409021795
Validation Accuracy (%): 86.77141409021795


In [15]:
processed_df = pd.DataFrame(X)

processed_df["Revenue"] = y.values

print(processed_df.shape)
print(processed_df.isnull().sum().sum())

(9864, 27)
0


In [16]:
model = LogisticRegression(max_iter=1000)

model.fit(X, y)

print("Final model trained successfully")

Final model trained successfully


In [17]:
predictions = model.predict(X_test)

print(predictions[:10])
print("Total predictions:", len(predictions))

[False False  True False False False False False False False]
Total predictions: 2466


In [18]:
import pandas as pd
import numpy as np

original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

row_retained_percent = (
    processed_rows / original_rows
) * 100

final_model = model

if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_

if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__

checkpoints = pd.DataFrame({

    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],

    "value": [
        original_missing,
        processed_missing,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns,
        round(row_retained_percent, 2),
        model_name
    ]
})

prediction_output = pd.DataFrame({

    "id": test_df["Session_ID"].astype(str),

    "value": np.asarray(predictions).astype(str)

})

submission = pd.concat(
    [checkpoints, prediction_output],
    ignore_index=True
)

submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully.")
print(checkpoints)

submission.csv created successfully.
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  27
6  row_retained_percent               100.0
7            model_name  LogisticRegression


In [19]:
submission = pd.read_csv("submission.csv")

print(submission.head(12))
print()
print("Submission shape:", submission.shape)
print("Unique IDs:", submission["id"].nunique())

                      id               value
0       original_missing                2957
1      processed_missing                   0
2          original_rows                9864
3         processed_rows                9864
4       original_columns                  19
5      processed_columns                  27
6   row_retained_percent               100.0
7             model_name  LogisticRegression
8                 106094               False
9                 111845               False
10                106794                True
11                103444               False

Submission shape: (2474, 2)
Unique IDs: 2474
